In [1]:
import logging
import sys
import socket
import logging
import math
import re
import glob
import time
import time
import os

from pathlib import Path
from itertools import cycle
from datetime import datetime

if sys.version_info[0] == 2:
    import Tkinter
    tkinter = Tkinter
    from Tkinter import *
    from Tkinter.ttk import *
    from Tkinter import filedialog 
else:
    import tkinter
    from tkinter import *
    from tkinter.ttk import *
    from tkinter import filedialog 


# from playsound import playsound
import cv2
import screeninfo
import numpy as np
from PIL import Image, ImageTk
from zaber_motion import Units
from zaber_motion.ascii import Connection
import sys
import usb.core
import usb.util 
from pprint import pprint
import dinglab_printer

In [4]:
# Check if running python 3
if sys.version_info[0] != 3:
    raise SystemExit("Please set kernel to Python 3.x.x")


In [5]:
from zaber_motion import Units
from zaber_motion.ascii import Connection

class ZaberMotor:
    def __init__(self):
          self.init = None

    def get_zaber_motor(self): 
        connection = Connection.open_serial_port("COM3")
        connection.enable_alerts()
        device_list = connection.detect_devices()
        print("Found {} devices".format(len(device_list)))
        device = device_list[0]
        axis = device.get_axis(1)
        return axis

In [2]:
# Test Connection to Zaber motors

# Test 1





# # find USB devices
# dev = usb.core.find(find_all=True)
# # loop through devices, printing vendor and product ids in decimal and hex
# for cfg in dev:
#     print('Decimal VendorID=' + str(cfg.idVendor) + ' & ProductID=' + str(cfg.idProduct) + '\n')
#     print('Hexadecimal VendorID=' + hex(cfg.idVendor) + ' & ProductID=' + hex(cfg.idProduct) + '\n\n')

# Test 2
# Check if zaber motor works
# z = ZaberMotor()
# z.get_zaber_motor()

# with Connection.open_serial_port("COM3") as connection:
#     connection.enable_alerts()
#     device_list = connection.detect_devices()
#     print("Found {} devices".format(len(device_list)))
#     device = device_list[0]
#     axis = device.get_axis(1)
#     print("Moving")
#     axis.move_absolute(25, Units.LENGTH_MILLIMETRES, False)
#     print("Moved")

In [3]:
## Test Laser printer connection

controller = dinglab_printer.Controller()
print(dir(controller))
f = controller.check_connection()
print(f.getSuccess())
print(f.getMessage())
f = controller.initialize()
print(f.getSuccess())
print(f.getMessage())


['__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'add', 'check_connection', 'initialize', 'test']
True
Connected Successfully
True
Projector is in READY...


In [36]:
class Application():
    def __init__(self):
        self.init = None
        
    def set_image_directory(self, txt_path=''):
        '''
        Layer	File	Thickness	Pause	Material	time	Intensity
        '''
        # txt_name = path.split('\\')[-1] +'.txt'
        path = os.path.dirname(txt_path)
        # print(' txt_name')
        # print(txt_name)
        # txt_path = Path(path).glob(txt_name)
        image_list = []
        exposure_time_list = []
        thickness_list = []
        
      
        with open(txt_path) as f:
            lines = f.readlines()
            for line in lines[1:]:
                elements = line.split()
                count, image_path, exposure_time, thickness = elements[0], elements[1], elements[2], elements[3]
                exposure_time_list.append(float(exposure_time))
                thickness_list.append(float(thickness))
                image_list.append(path +'\\' + image_path)
#             self.images = iter(map(ImageTk.PhotoImage, map(Image.open, iter(image_list))))
#             self.duration_ms_list = iter(iter(exposure_time_list))
        return image_list, exposure_time_list, thickness_list
    
    def generate_debug_txt(self, path='', thickness='5', pause='0', material='1', time='1', intensity='0'):
        txt_name = path.split('\\')[-1] + '.txt'
        txt_path = path + '\\'+ txt_name
        image_paths = Path(path).glob("*[!.txt]")
        file_pattern = re.compile(r'.*?(\d+).*?')
        def get_order(file):
            match = file_pattern.match(Path(file).name)
            if not match:
                return math.inf
            return int(match.groups()[-1])
        image_paths = sorted(image_paths, key=get_order)
        try:
            with open(txt_path, 'w') as f:
                f.write('Layer	File	Thickness	Pause	Material	Time	Intensity\n')
                layer = 1
                while image_paths:
                    image_name = str(image_paths.pop(0)).split('\\')[-1]
                    line = str(layer) + '   ' + image_name + '  ' + thickness \
                           + '  ' + pause + '  ' + material + '  ' + time + '  ' + intensity + '\n'
                    f.write(line)
                    layer += 1
        except FileNotFoundError:
            print("The directory does not exist for creating the text file.")

In [ ]:


class MyWindow:
    def __init__(self, win):
        instruction = '''
Check List:\n
1. Make sure the DLP Lightcrafter GUI is closed.\n
2. Open A3200 Motion Composer and all three axes are connected.\n

Trouble Shooting:\n
1. USB Input/output error: Close DLP Lightcrafter GUI.\n
2. Reconnect stage with A3200 Motion Composer.\n
'''
        credit = '''
Professor Cheng Sun
Boyuan Sun, boyuansun2026@u.northwestern.edu
Evan Jones, XXX@northwestern.edu
Edwin Clement, eclement@wpi.edu
'''
        self.reference = 0
        self.image_list = []
        self.exposure_time = []
        self.thickness = []
        self.win = win
        self.flag = False
        self.offset = -25
        
        self.canvas1 = Canvas(
        win,
        height=200,
        width=270,
        bg="#FFEFD5"
        )
        self.canvas1.place(x=70, y=520)
        self.canvas2 = Canvas(
        win,
        height=200,
        width=270,
        bg="#FFEFD5"
        )
        self.canvas2.place(x=370, y=520)
        
        self.lbl0 = Label(win, text='Rush', font='Helvetica 40 bold')
        self.lbl1 = Label(win, text='Directory of Images')
#         self.lbl2 = Label(win, text='Layer Thickness(um)')
#         self.lbl3 = Label(win, text='Exposure Time(s)')
        self.lbl4 = Label(win, text='Z Axis Position')
        self.lbl5 = Label(win, text=instruction, font='Helvetica 10',foreground='purple')
        self.lbl6 = Label(win, text=credit, font='Helvetica 7')
        self.lbl7 = Label(win, text='Printing Progress')
        self.lbl8 = Label(win, text='System Message:')
        self.lbl9 = Label(win, text='Move distance(mm)')
        self.lbl10 = Label(win, text='Layer thickness(um)')
        self.lbl11 = Label(win, text='Exposure time(s)')
        self.lbl12 = Label(win, text='Stage Control', font='Helvetica 12 bold')
        self.lbl13 = Label(win, text='Simple Txt File Generator', font='Helvetica 12 bold')
        self.lbl14 = Label(win, text='LED Current(0-255)')
        # self.t1 = Entry(width=160)
#         self.t2 = Entry()
#         self.t3 = Entry()
        self.t4 = Entry()
        self.t8 = Entry()
        self.t9 = Entry()
        self.t10 = Entry()
        self.t11 = Entry()
        self.t14 = Entry()
        self.lbl0.place(x=550, y=50)
        self.lbl1.place(x=50, y=150)
        # self.t1.place(x=180, y=150)
#         self.lbl2.place(x=50, y=200)
#         self.t2.place(x=180, y=200)
#         self.lbl3.place(x=370, y=200)
#         self.t3.place(x=500, y=200)
        self.lbl4.place(x=50, y=260)
        self.t4.place(x=50, y=280)
        self.lbl5.place(x=700, y=270)
        self.lbl6.place(x=950, y=0)
        self.t8.place(x=500, y=280)
        self.lbl8.place(x=500, y=260)
        self.t9.place(x=140, y=580)
        self.lbl9.place(x=140, y=560)
        self.t10.place(x=400, y=580)
        self.lbl10.place(x=400, y=560)
        self.t11.place(x=400, y=620)
        self.lbl11.place(x=400, y=600)
        self.lbl12.place(x=150, y=500)
        self.lbl13.place(x=410, y=500)
        self.t14.place(x=240, y=280)
        self.lbl14.place(x=240, y=260)

        self.progress = Progressbar(win, orient=HORIZONTAL, length=500, mode='determinate')
        self.progress.place(x=50, y=430)
        self.lbl7.place(x=250, y=400)
        
        def setFilePath():
            file_selected = filedialog.askopenfilename(filetypes=(("3D Slices Info", '*.txt'),))
            self.filePath.set(os.path.normpath(file_selected))
            self.input_directory()
            avg_exp = sum(self.exposure_time)/len(self.exposure_time)
            avg_thk = sum(self.thickness)/len(self.thickness)
            
            self.printJobDetails.set(f'Cnt: {len(self.image_list)} Avg. Exp {round(avg_exp,2)}, Avg. Thk {round(avg_thk,2)}')

            
            
            
        self.filePath = StringVar()
        self.printJobDetails =  StringVar()
        # self.lblName = Label(self, text='Select File with Image Lists')
        self.entPath = Entry(win, width=120, textvariable=self.filePath)
        self.btnFind = Button(win, text="Select File",command=setFilePath)
        self.loaded_image_info = Label(win, textvariable=self.printJobDetails)
        self.printJobDetails.set('Print Job Details')

        self.entPath.place(x=180, y=150)
        self.btnFind.place(x=180+730+10,y=150)      
        self.loaded_image_info.place(x=180+740+70+10,y=150)     
        self.entPath.insert(END, str("E:\_Animal_Study_Stent_ADJ3_Henry_50um\sent_50um.txt"))

        self.b1 = Button(win, text='Run', command=self.run)
        self.b2 = Button(win, text='Set Home', command=self.set_home)
        self.b3 = Button(win, text='Get Position', command=self.get_position)
        self.b4 = Button(win, text='Stop', command=self.stop)
        self.b5 = Button(win, text='Move Down', command=self.movedown)
        self.b6 = Button(win, text='Move Up', command=self.moveup)
        self.b7 = Button(win, text='Simple input txt generator', command=self.simple_txt)
        self.b8 = Button(win, text='Set Power', command=self.set_power)
        
        self.b1.place(x=70, y=200)
        self.b2.place(x=50, y=310)
        self.b3.place(x=130, y=310)
        self.b4.place(x=170, y=200)
        self.b5.place(x=100, y=630)
        self.b6.place(x=200, y=630)
        self.b7.place(x=440, y=660)
#         self.b8.place(x=240, y=310)
        
#         self.controller = pycrafter9000.dmd()


        self.application = Application()
        # self.controller.stopsequence()
#         self.controller.changemode(3)
        
        # ip = 'localhost'
        # port = 8000
        # self.my_ensemble = Ensemble(ip, port)
        # self.my_ensemble.connect()
        # self.my_ensemble.write_read('BLOCKMOTION X Y 1')
        # self.my_ensemble.write_read('BLOCKMOTION Z 0')
        # self.my_ensemble.write_read('ENABLE Z')
        
#         # TODO 
#         self.axis = zaberMotor().get_zaber_motor()
        
        # self.t1.delete(0, 'end')
        self.t4.delete(0, 'end')
        self.t8.delete(0, 'end')
        self.t9.delete(0, 'end')
        self.t10.delete(0, 'end')
        self.t11.delete(0, 'end')
        self.t14.delete(0, 'end')
        self.t4.insert(END, str("0"))
        self.t8.insert(END, str("Stage connected"))
        self.t9.insert(END, str("25"))
        self.t10.insert(END, str("5"))
        self.t11.insert(END, str("1.5"))
        self.t14.insert(END, str("100"))
        
        screen_id = 0
        self.screen = screeninfo.get_monitors()[screen_id]
        print("Screen info")
        print(screeninfo.get_monitors())
        self.window_name = 'show'
        self.black_image = np.zeros((1080,1920))
        
    def run(self):
        """
        Perform a print
        """    
        self.initilze_stage()
        self.controller.stopsequence()
        self.controller.changemode(3)
        power = int(self.t14.get())
        self.controller.power(current=power)
        self.flag = True
        # self.my_ensemble.write_read('MOVEABS Z {} 2'.format(self.reference))
        self.axis.move_absolute(self.reference, Units.LENGTH_MILLIMETRES)
        
        cv2.namedWindow(self.window_name, cv2.WND_PROP_FULLSCREEN)
        # get the screen dimensions and properly center the image
        cv2.moveWindow(self.window_name, self.screen.x +1439, self.screen.y -1)
        cv2.setWindowProperty(self.window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
        cv2.imshow(self.window_name, self.black_image)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            cv2.destroyAllWindows()
            
        # while '4202508' != self.my_ensemble.write_read('AXISSTATUS(Z, DATAITEM_AxisStatus)'):
        while self.axis.is_busy():
            time.sleep(0.2)
        # self.controller.changemode(0)
        self._(0)
        # self.controller.changemode(3)
        # self.my_ensemble.write_read('MOVEINC Z {} 3'.format(self.offset))
        self.axis.move_relative(self.offset, Units.LENGTH_MILLIMETRES, True)
        cv2.destroyAllWindows()
        # while '4202508' != self.my_ensemble.write_read('AXISSTATUS(Z, DATAITEM_AxisStatus)'):
            # time.sleep(0.2)
#         self.t8.delete(0, 'end')
#         self.t8.insert(END, str("Print Done"))
        
    def _(self, idx):
        image = cv2.imread(self.image_list[idx].replace('\\','\\\\'), cv2.IMREAD_GRAYSCALE)
#         print(self.image_list)
#         print(self.image_list[idx].replace('\\','\\\\'))
        cv2.imshow(self.window_name, image)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            cv2.destroyAllWindows()
        # self.my_ensemble.write_read('MOVEINC Z {} {}'.format((self.thickness[idx]*-1)/1000, self.thickness[idx]/self.exposure_time[idx]))
        self.axis.move_relative((self.thickness[idx]*-1)/1000, Units.LENGTH_MILLIMETRES, True, self.thickness[idx]/self.exposure_time[idx], Units.VELOCITY_MILLIMETRES_PER_SECOND)
        time.sleep(self.exposure_time[idx])
        cv2.imshow(self.window_name, self.black_image)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            cv2.destroyAllWindows()
        idx += 1
        self.progress['value'] = 100/len(self.exposure_time)*idx
        if idx >= len(self.exposure_time):
            self.flag = False
        if self.flag:
            self.win.update()
            self.win.after(5, self._(idx))


            
    def set_home(self):
        """
        Set the position to home
        """
        self.reference = float(self.t4.get())
        # self.my_ensemble.write_read('MOVEINC Z -25 3'.format(self.offset))
        self.axis.move_min(True)
        self.t8.delete(0, 'end')
        self.t8.insert(END, str("Home Set"))
        
    def get_position(self):
        """
        Update Current Z Position
        :return:
        """
        self.t4.delete(0, 'end')
        # self.reference = float(self.my_ensemble.write_read('AXISSTATUS(Z, DATAITEM_PositionCommand)'))
        self.reference = float(self.axis.get_position(Units.LENGTH_MILLIMETRES))
        self.t4.insert(END, str(self.reference))
        
    def stop(self):
        """
        User Interruption
        :return:
        """
        self.controller.stopsequence()
        self.controller.changemode(3)
        self.flag = False
        cv2.destroyAllWindows()
        
    def initilze_stage(self):
        # self.my_ensemble.write_read('BLOCKMOTION X Y 1')
        # self.my_ensemble.write_read('BLOCKMOTION Z 0')
        # self.my_ensemble.write_read('ENABLE Z')
        self.axis.move_min(True)
        
    def input_directory(self):
        """
        Input all images from txt
        """
        path = str(self.entPath.get())
#         self.application.generate_debug_txt(path)
        self.image_list, self.exposure_time, self.thickness = self.application.set_image_directory(path)
        
    def moveup(self):
        """
        Move up by distance(mm) given
        """
        # self.my_ensemble.write_read('BLOCKMOTION X Y 1')
        # self.my_ensemble.write_read('BLOCKMOTION Z 0')
        # self.my_ensemble.write_read('ENABLE Z')
        # self.my_ensemble.write_read('MOVEINC Z {} 3'.format(float(self.t9.get())*-1))
        self.axis.move_relative((float(self.t9.get())*-1), Units.LENGTH_MILLIMETRES, False)
        
    def movedown(self):
        """
        Move up by distance(mm) given
        """
        # self.my_ensemble.write_read('BLOCKMOTION X Y 1')
        # self.my_ensemble.write_read('BLOCKMOTION Z 0')
        # self.my_ensemble.write_read('ENABLE Z')
        # self.my_ensemble.write_read('MOVEINC Z {} 3'.format(self.t9.get()))
        self.axis.move_relative((float(self.t9.get())), Units.LENGTH_MILLIMETRES, False)
        
    def simple_txt(self):
        """
        Generator txt with given exposure time and layer thickness
        """
        path = str(self.entPath.get())
        thickness = str(self.t10.get())
        time = str(self.t11.get())
        self.application.generate_debug_txt(path=path,thickness=thickness,time=time)
        
    def set_power(self):
        """
        Generator txt with given exposure time and layer thickness
        """
        power = int(self.t14.get())
        self.controller.stopsequence()
        self.controller.power(current=power)

window = Tk()
mywin = MyWindow(window)
window.title('')
window.geometry("1200x800+10+10")
window.mainloop()

Screen info
[Monitor(x=0, y=0, width=1920, height=1080, width_mm=477, height_mm=268, name='\\\\.\\DISPLAY1', is_primary=True)]
